# Precompute NNI delta caches

This notebook caches the expensive NNI delta precomputation used by downstream analyses. It follows the same pattern as the equal-edge notebook: if the cache already exists it is loaded, and if not it is computed and saved to `cached_data/` unless `FORCE_RECOMPUTE` is set to `True`.

This takes ~5 minutes for 100 trees. 

In [16]:
import sys
from pathlib import Path

def get_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / "src").exists() and (cwd / "notebooks").exists():
        return cwd
    if cwd.name == "notebooks" and (cwd.parent / "src").exists():
        return cwd.parent
    if cwd.parent.name == "notebooks" and (cwd.parent.parent / "src").exists():
        return cwd.parent.parent
    raise RuntimeError("Could not infer the repository root from the current working directory.")

REPO_ROOT = get_repo_root()
SRC_DIR = REPO_ROOT / "src"
CACHE_DIR = REPO_ROOT / "cached_data"
CACHE_DIR.mkdir(exist_ok=True)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from inconsistency_generation.nni_delta_precomputation import (
    default_flip_weights,
    default_nni_delta_cache_path,
    load_or_compute_nni_delta_bundle,
)

In [17]:
N_LEAVES = [5, 6, 7, 8, 9]
N_TREES = 500
ALPHA = 0.20
BETA = 0.01
SEED = 42
FORCE_RECOMPUTE = True

W_POS, W_NEG = default_flip_weights(ALPHA, BETA)
CACHE_PATH = default_nni_delta_cache_path(
    cache_dir=CACHE_DIR,
    n_leaves_list=N_LEAVES,
    n_trees=N_TREES,
    alpha=ALPHA,
    beta=BETA,
    seed=SEED,
    w_pos=W_POS,
    w_neg=W_NEG,
)
CACHE_PATH

PosixPath('/Users/satasg/Documents/repos/inconsistency_repo/cached_data/nni_deltas_n5-6-7-8-9_trees500_a0.200_b0.010_seed42.pkl')

## Load Or Compute

This will load the cached bundle if it already exists. Otherwise it will generate the full bundle for all requested values of `N_LEAVES` and save it into `cached_data/`.
This 

In [18]:
bundle = load_or_compute_nni_delta_bundle(
    cache_path=CACHE_PATH,
    n_leaves_list=N_LEAVES,
    n_trees=N_TREES,
    alpha=ALPHA,
    beta=BETA,
    seed=SEED,
    w_pos=W_POS,
    w_neg=W_NEG,
    force_recompute=FORCE_RECOMPUTE,
    verbose=True,
)

[n=5] Generating 500 trees and precomputing NNI deltas...
  10/500 trees (0.1s elapsed)
  20/500 trees (0.2s elapsed)
  30/500 trees (0.3s elapsed)
  40/500 trees (0.4s elapsed)
  50/500 trees (0.5s elapsed)
  60/500 trees (0.6s elapsed)
  70/500 trees (0.7s elapsed)
  80/500 trees (0.8s elapsed)
  90/500 trees (0.9s elapsed)
  100/500 trees (1.0s elapsed)
  110/500 trees (1.1s elapsed)
  120/500 trees (1.2s elapsed)
  130/500 trees (1.3s elapsed)
  140/500 trees (1.3s elapsed)
  150/500 trees (1.4s elapsed)
  160/500 trees (1.5s elapsed)
  170/500 trees (1.6s elapsed)
  180/500 trees (1.7s elapsed)
  190/500 trees (1.8s elapsed)
  200/500 trees (1.9s elapsed)
  210/500 trees (2.0s elapsed)
  220/500 trees (2.1s elapsed)
  230/500 trees (2.2s elapsed)
  240/500 trees (2.3s elapsed)
  250/500 trees (2.4s elapsed)
  260/500 trees (2.5s elapsed)
  270/500 trees (2.6s elapsed)
  280/500 trees (2.7s elapsed)
  290/500 trees (2.7s elapsed)
  300/500 trees (2.8s elapsed)
  310/500 trees (2.9s

In [19]:
summary_rows = []
for n_leaves, tree_records in bundle["results_by_n"].items():
    neighbor_records = sum(len(tree_record["nni_data"]) for tree_record in tree_records)
    summary_rows.append(
        {
            "n_leaves": n_leaves,
            "n_trees": len(tree_records),
            "n_neighbor_records": neighbor_records,
        }
    )

summary_rows

[{'n_leaves': 5, 'n_trees': 500, 'n_neighbor_records': 2526},
 {'n_leaves': 6, 'n_trees': 500, 'n_neighbor_records': 3624},
 {'n_leaves': 7, 'n_trees': 500, 'n_neighbor_records': 4678},
 {'n_leaves': 8, 'n_trees': 500, 'n_neighbor_records': 5728},
 {'n_leaves': 9, 'n_trees': 500, 'n_neighbor_records': 6772}]

In [20]:
example_n = N_LEAVES[0]
example_tree_record = bundle["results_by_n"][example_n][0]
{
    "n_leaves": example_n,
    "tree_edge_count": len(example_tree_record["tree_star"]),
    "neighbor_count": len(example_tree_record["nni_data"]),
    "neighbor_record_keys": list(example_tree_record["nni_data"][0].keys()),
}

{'n_leaves': 5,
 'tree_edge_count': 9,
 'neighbor_count': 4,
 'neighbor_record_keys': ['nni_edge',
  'nni_index',
  'split_size',
  'abc_sizes',
  'deltas']}